# Robot manipulation with VLA-JEPA and OpenVINO

[VLA-JEPA](https://github.com/ginwind/VLA-JEPA) is a Vision-Language-Action (VLA) policy for
robot manipulation. Unlike a chatbot VLM, which emits text, a VLA policy emits **continuous robot
actions**: given camera images and a natural-language instruction such as *"pick up the black bowl
and place it on the plate"*, it predicts the motion needed to carry out the task.

VLA-JEPA is built from two components:

1. **A Qwen3-VL-2B vision-language backbone.** Camera views and the instruction are packed into a
   single prompt. The prompt reserves 32 placeholder positions (`<|embodied_action|>`) whose final
   hidden states become a compact, action-oriented summary of the scene — the *embodied action
   tokens*. The backbone is used purely as an encoder here; nothing is ever decoded to text.
2. **A DiT flow-matching action head.** A small (150M) diffusion transformer conditions on those 32
   tokens plus the robot's own internal state readings and denoises a noise sample into a 7-step action
   chunk over 4 Euler steps. Predicting a *chunk* rather than a single step is what gives the policy
   temporal consistency.

```
   camera views ──┐
                  ├──► Qwen3-VL-2B ──► 32 embodied ──► DiT action head ──► 7 × 7-DoF
   instruction ───┘      (encoder)     action tokens    (4 flow-matching     action chunk
                                                          Euler steps)
                              robot state ───────────────────┘
```

Each of the 7 predicted actions is a 7-DoF vector: 3 translation, 3 rotation, and 1 binary gripper
open/close command.

In this tutorial we build a PyTorch reference from the checkpoint, convert both components to
OpenVINO Intermediate Representation, run the converted pipeline on a sample observation, and check
stage by stage that the conversion preserved the policy.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Build the pipeline config and sample observation](#Build-the-pipeline-config-and-sample-observation)
- [Generate the PyTorch reference](#Generate-the-PyTorch-reference)
- [Convert the model to OpenVINO IR](#Convert-the-model-to-OpenVINO-IR)
    - [Why the action head is re-implemented](#Why-the-action-head-is-re-implemented)
    - [Why the backbone needs its final norm removed](#Why-the-backbone-needs-its-final-norm-removed)
- [Select inference device](#Select-inference-device)
- [Load the OpenVINO pipeline](#Load-the-OpenVINO-pipeline)
- [Run inference on a sample observation](#Run-inference-on-a-sample-observation)
- [Validate against the PyTorch reference](#Validate-against-the-PyTorch-reference)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/vlajepa/vla-jepa.ipynb" />

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

> **Note:** `transformers` is pinned to 4.57.x. The VLA-JEPA checkpoint was trained and validated
> against that version, and installing `optimum-intel` unpinned pulls `transformers` 5.x, which
> changes the Qwen3-VL graph enough to break parity with the reference outputs.

In [ ]:
%pip install -q "torch>=2.6" "torchvision" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q "openvino>=2025.0" "nncf>=2.14"
%pip install -q "optimum-intel[openvino]" "transformers==4.57.*"
%pip install -q "diffusers<0.38" "qwen-vl-utils" "omegaconf" "Pillow" "pandas" "ipywidgets"
%pip install -q "gradio>=4.19,<6" "huggingface-hub<1.0,>=0.34.0"

In [ ]:
from pathlib import Path

import requests

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    open("notebook_utils.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("vla-jepa.ipynb")

The notebook needs the fine-tuned VLA-JEPA checkpoint and the Qwen3-VL-2B base model it was
built on. The cell below checks for them under `pretrained/` and downloads whatever is missing
from the Hugging Face Hub.

In [ ]:
import numpy as np
import openvino as ov
from pathlib import Path
import vlajepa_helper as vlajepa

core = ov.Core()

# Everything the notebook writes stays next to the notebook.
MODEL_DIR = Path("model")
REFERENCE_DIR = Path("reference")

# Pre-trained VLA-JEPA assets. Adjust these three paths if your copies live elsewhere.
CHECKPOINT = Path("pretrained/LIBERO/checkpoints/VLA-JEPA-LIBERO.pt")
CONFIG_YAML = Path("pretrained/LIBERO/config.yaml")
DATASET_STATS = Path("pretrained/LIBERO/dataset_statistics.json")
QWEN_BASE = Path("pretrained/Qwen3-VL-2B-Instruct")

missing = [p for p in (CHECKPOINT, CONFIG_YAML, DATASET_STATS, QWEN_BASE) if not p.exists()]
if missing:
    import download_checkpoints

    print("Missing VLA-JEPA assets, downloading them now:\n  " + "\n  ".join(str(p) for p in missing))
    targets = []
    if not QWEN_BASE.exists():
        targets.append("qwen")
    if any(not p.exists() for p in (CHECKPOINT, CONFIG_YAML, DATASET_STATS)):
        targets.append("libero")
    for target in targets:
        download_checkpoints.DOWNLOADERS[target](Path("pretrained"))

for name, path in [("checkpoint", CHECKPOINT), ("train config", CONFIG_YAML), ("dataset stats", DATASET_STATS), ("Qwen3-VL base", QWEN_BASE)]:
    assert path.exists(), f"Still missing {name} at {path}"
    print(f"  OK  {name:15s} {path}")

## Build the pipeline config and sample observation
[back to top ⬆️](#Table-of-contents:)

Both converted components and the inference pipeline are driven by a small config: prompt template,
token ids, action/state dimensions, the flow-matching schedule, and the action normalization
statistics from training. All of it is already determined by the training config and statistics
shipped with the checkpoint, so `vlajepa_helper.build_config` derives it directly rather than
requiring a config file to be supplied.

The sample observation is a deterministic LIBERO-style input, two 224×224 RGB views plus an
8-dimensional robot-state vector, drawn from a fixed seed, so every number below is reproducible
run to run and across machines.

In [ ]:
from PIL import Image

INSTRUCTION = "pick up the black bowl and place it on the plate"

cfg = vlajepa.build_config(CONFIG_YAML, DATASET_STATS, instruction=INSTRUCTION)
images, images_np, state = vlajepa.make_fixed_input(cfg)

print(f"action chunk        : {cfg['action_horizon']} steps x {cfg['action_dim']} DoF")
print(f"flow-matching steps : {cfg['num_inference_timesteps']}  (timesteps {cfg['timesteps']})")
print(f"embodied tokens     : {cfg['num_embodied_action_tokens']} x {cfg['qwen_hidden_size']}")
print(f"instruction         : {INSTRUCTION}")
print(f"robot state         : {np.round(state[0], 3)}")

display(Image.fromarray(np.hstack(list(images_np))))

## Generate the PyTorch reference
[back to top ⬆️](#Table-of-contents:)

Before converting anything we run the pipeline in PyTorch and keep every intermediate tensor: the 32 embodied action
tokens, the noisy actions and predicted velocity at each of the 4 flow-matching steps, and the final
action chunk.

The reference is rebuilt from the checkpoint using only `transformers` and the action-head definition
in `export.py`. The upstream project produces these tensors by
importing the VLA-JEPA training tree, which additionally pulls in a V-JEPA 2 encoder that inference
never touches; rebuilding just the inference path keeps this notebook standalone and compares
like with like.

This loads the 2B backbone in FP32 on CPU and takes a few minutes.

In [ ]:
if (REFERENCE_DIR / "pred_actions.npy").exists():
    print(f"Reference tensors already present in {REFERENCE_DIR} — skipping.")
else:
    vlajepa.generate_reference(REFERENCE_DIR, cfg, CHECKPOINT, CONFIG_YAML, QWEN_BASE)

reference = {p.stem: np.load(p) for p in sorted(REFERENCE_DIR.glob("*.npy"))}
for name in ["embodied_action_tokens", "initial_noise", "pred_actions", "unnormalized_actions"]:
    print(f"  {name:24s} {reference[name].shape}")

## Convert the model to OpenVINO IR
[back to top ⬆️](#Table-of-contents:)

The two components are converted by different routes, because they are different kinds of model:

| Component | Route | Result |
|---|---|---|
| Qwen3-VL-2B backbone | [Optimum Intel](https://huggingface.co/docs/optimum/intel/index) `export_from_model` | 5 IRs under `model/qwen3vl/` |
| DiT action head | `ov.convert_model` on a traced module | `model/action_dit.xml` |

Qwen3-VL is already enabled in Optimum Intel, which splits it into five IRs (vision embeddings,
positional embeddings, vision merger, text embeddings, and the language model) and wires them
together behind a single class. VLA-JEPA only ever runs one prefill pass and never decodes token by token, so the cache is write-only.

In [ ]:
import subprocess
import sys

if (MODEL_DIR / "action_dit.xml").exists() and (MODEL_DIR / "qwen3vl/openvino_language_model.xml").exists():
    print(f"IRs already present in {MODEL_DIR} — skipping conversion.")
else:
    cmd = [
        sys.executable,
        "export.py",
        "--checkpoint",
        str(CHECKPOINT),
        "--config",
        str(CONFIG_YAML),
        "--qwen-base",
        str(QWEN_BASE),
        "--output-dir",
        str(MODEL_DIR),
        # The reference directory doubles as the source of `config.json`, which the
        # exporter copies next to the IRs for the inference pipeline to read, and as
        # the tensors the DiT conversion is checked against on the way out.
        "--golden-dir",
        str(REFERENCE_DIR),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

In [ ]:
for xml in sorted(MODEL_DIR.rglob("*.xml")):
    mb = xml.with_suffix(".bin").stat().st_size / 2**20 if xml.with_suffix(".bin").exists() else 0.0
    print(f"  {str(xml.relative_to(MODEL_DIR)):48s} {mb:9.1f} MB")

## Select inference device
[back to top ⬆️](#Table-of-contents:)

Select the device from the dropdown list to run inference on. Both `CPU` and an Intel GPU are
supported; `CPU` is useful as an exact-precision reference.

In [ ]:
from notebook_utils import device_widget

device = device_widget(default="AUTO", exclude=["NPU"])

device

## Load the OpenVINO pipeline
[back to top ⬆️](#Table-of-contents:)

`VlaJepaOV` re-implements the policy's `predict_action` on top of the converted IRs. It loads the
five backbone IRs through `OVModelForVisualCausalLM`, compiles the DiT head, and exposes the pipeline
in three stages, `preprocess`, `encode`, `denoise`, so each can be timed and validated separately.

In [ ]:
from run_inference_standalone import VlaJepaOV, unnormalize_actions

policy = VlaJepaOV(MODEL_DIR, device.value)

## Run inference on a sample observation
[back to top ⬆️](#Table-of-contents:)

We reuse the fixed two-view observation built earlier, so the OpenVINO result can be compared
directly against the PyTorch reference.

In [ ]:
inputs = policy.preprocess(images, INSTRUCTION)
input_ids = np.asarray(inputs["input_ids"])
n_embodied = int((input_ids == policy.emb_token_id).sum())

print(f"prompt length            : {input_ids.shape[1]} tokens")
print(f"embodied action tokens   : {n_embodied}")
print()
print(policy.build_prompt(INSTRUCTION)[:300].replace("<|action_0|>" * 8, "<|action_0|> x8 ") + " …")

Now run the two model stages. The backbone produces a `[1, 32, 2048]` conditioning tensor; the head
denoises that into a `[1, 7, 7]` action chunk. A fixed initial noise sample is injected so the result
is reproducible and directly comparable to the reference.

In [ ]:
embodied_tokens = policy.encode(inputs)
normalized_actions = policy.denoise(embodied_tokens, state, initial_noise=reference["initial_noise"])

print(f"embodied action tokens : {embodied_tokens.shape}")
print(f"normalized actions     : {normalized_actions.shape}")

The network predicts actions in a normalized space. Converting them back to physical units uses the
1st/99th percentile statistics from training, and the gripper column uses a threshold of 0.5 rather
than scaled, since it is a discrete open/close command, not a continuous value.

In [ ]:
import pandas as pd

actions = unnormalize_actions(normalized_actions[0].copy(), policy.cfg["action_norm_stats"])

pd.DataFrame(
    np.round(actions, 4),
    columns=["x", "y", "z", "roll", "pitch", "yaw", "gripper"],
    index=[f"t+{i}" for i in range(actions.shape[0])],
)

Each row is one future timestep of the chunk. A robot controller would execute some or all of these
before re-planning; the columns are translation (`x`, `y`, `z`), rotation (`roll`, `pitch`, `yaw`),
and the binary gripper command.

## Validate against the PyTorch reference
[back to top ⬆️](#Table-of-contents:)

Now we can answer whether the conversion changed the model, by comparing each stage against the
tensors captured from the PyTorch run.

A policy has no accuracy score of its own here, so we report **cosine similarity**, stage by stage,
which is what the downstream consumer of each tensor actually depends on. Two of the gates deserve
an explanation:

* **The embodied action tokens are scored at a looser gate than everything after them.** This is the
  pre-norm hidden state, and Qwen carries *massive activations* on it: a few of the 2048 channels
  reach magnitude ~80 against a typical per-token RMS of ~2.5, and they dominate the cosine. A small
  relative error on those channels moves the metric far more than it moves anything downstream —
  which is why the action outputs score tighter than their own input does.
* **The gripper column is scored separately, and not by cosine.** It is a hard 0/1 decision taken at
  a 0.5 threshold, so folding it into a cosine over continuous columns lets a single flipped bit
  swamp a metric that is otherwise reporting six well-behaved columns. Bits are compared exactly,
  but only where the reference clears the threshold by a margin — a step that lands within rounding
  distance of 0.5 is a coin flip that measures the sample, not the port.

In [ ]:
cos_sim = vlajepa.cos_sim
rows = []

rows.append(("embodied action tokens", cos_sim(reference["embodied_action_tokens"], embodied_tokens), 0.995))

# Each velocity is driven by the REFERENCE noisy actions rather than the OpenVINO ones, so a
# step is judged on its own rather than on drift accumulated by the steps before it.
emb_ref = np.ascontiguousarray(reference["embodied_action_tokens"], dtype=np.float32)
state_ref = np.asarray(state, np.float32).reshape(1, 1, cfg["state_dim"])
for i, t_disc in enumerate(cfg["timesteps"]):
    velocity = policy.dit(
        {
            "noisy_actions": np.ascontiguousarray(reference[f"dit_noisy_actions_step{i}"], np.float32),
            "timestep": np.array([t_disc], dtype=np.int64),
            "embodied_tokens": emb_ref,
            "state": state_ref,
        }
    )["velocity"]
    rows.append((f"velocity t={t_disc}", cos_sim(reference[f"dit_velocity_step{i}"], velocity), 0.999))

rows.append(("predicted actions", cos_sim(reference["pred_actions"], normalized_actions), 0.990))
# Continuous columns only — the gripper bit is checked exactly, below.
rows.append(("unnormalized actions (cols 0-5)", cos_sim(reference["unnormalized_actions"][:, :6], actions[:, :6]), 0.990))

print(f"{'stage':<34}{'cosine':>12}{'gate':>8}   result")
print("-" * 66)
all_pass = True
for name, value, gate in rows:
    all_pass &= value >= gate
    print(f"{name:<34}{value:>12.6f}{gate:>8.3f}   {'PASS' if value >= gate else 'FAIL'}")

ref_normalized_gripper = reference["pred_actions"][0][:, 6]
decisive = np.abs(ref_normalized_gripper - 0.5) >= 0.05
gripper_ok = np.array_equal(actions[decisive, 6], reference["unnormalized_actions"][decisive, 6])
all_pass &= gripper_ok
print("-" * 66)
print(
    f"gripper bits {actions[:, 6].astype(int)} vs reference "
    f"{reference['unnormalized_actions'][:, 6].astype(int)}   "
    f"{'PASS' if gripper_ok else 'FAIL'} ({int(decisive.sum())}/{decisive.size} decisive)"
)
for i in np.nonzero(~decisive)[0]:
    print(
        f"  step {i} not gated: reference normalized {ref_normalized_gripper[i]:.4f} is only "
        f"{abs(ref_normalized_gripper[i] - 0.5):.4f} from the 0.5 threshold"
    )
print("-" * 66)